In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base_dir = "/content/drive/MyDrive/huggingface-rag"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -U sentence-transformers accelerate einops

In [4]:
import json
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

In [6]:
model_name = "jinaai/jina-embeddings-v2-base-en"
max_seq_length = 512
batch_size = 8
num_epochs = 1
output_path = f"{base_dir}/ft-jina-transformers-v1"

model = SentenceTransformer(model_name, trust_remote_code=True)
model.max_seq_length = max_seq_length

In [7]:
train_examples = []
dataset_path = f"{base_dir}/train_dataset.jsonl"

with open(dataset_path, 'r', encoding='utf-8') as f_in:
  for line in f_in:
    item = json.loads(line)
    train_examples.append(InputExample(texts=[
      item['query'],
      item['pos'],
      item['neg']
    ]))

print(f"{len(train_examples)} train data is loaded")

15019 train data is loaded


In [6]:
train_dataloader = DataLoader(
  train_examples,
  shuffle=True,
  batch_size=batch_size
)

train_loss = losses.MultipleNegativesRankingLoss(model=model)

print("Start finetuning...\n")

model.fit(
  train_objectives=[(train_dataloader, train_loss)],
  epochs = num_epochs,
  warmup_steps=int(len(train_dataloader) * 10),
  output_path=output_path,
  show_progress_bar=True,
  use_amp=True
)

print(f"Finetune over! Finetuned model has been saved in {output_path}")

Start finetuning...



Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: fuy60703 (fuy60703-huazhong-university-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.980200
1000,0.278000
1500,0.202600


Finetune over! Finetuned model has been saved in /content/drive/MyDrive/huggingface-rag/ft-jina-transformers-v1


In [8]:
import json
import random
import numpy as np
from sentence_transformers import util

model = SentenceTransformer(output_path, trust_remote_code=True)
model.to('cpu')
model.eval()

# Load dataset
with open(dataset_path, 'r', encoding='utf-8') as f:
  all_data = [json.loads(line) for line in f]

# Evaluation on sample pairs
print("Starting fine-tuning evaluation...\n")
print("=" * 100)

test_samples = random.sample(all_data, min(40, len(all_data)))
correct_predictions = 0

for idx, sample in enumerate(test_samples, 1):
  query = sample['query']
  pos_doc = sample['pos']
  neg_doc = sample['neg']

  # All embeddings are CPU numpy
  query_embedding = model.encode(query, convert_to_tensor=False)
  pos_embedding = model.encode(pos_doc, convert_to_tensor=False)
  neg_embedding = model.encode(neg_doc, convert_to_tensor=False)

  # Convert to torch tensor for similarity
  query_tensor = torch.tensor(query_embedding)
  pos_tensor = torch.tensor(pos_embedding)
  neg_tensor = torch.tensor(neg_embedding)

  pos_score = util.cos_sim(query_tensor, pos_tensor).item()
  neg_score = util.cos_sim(query_tensor, neg_tensor).item()
  is_correct = pos_score > neg_score
  correct_predictions += is_correct

  print(f"\nSample {idx}/{len(test_samples)}")
  print(f"Positive: {pos_score:.4f}, Negative: {neg_score:.4f}")
  print(f"Result: {'✓ Correct' if is_correct else '✗ Wrong'}")
  print("-" * 90)

print("\n" + "=" * 100)
print(f"Pairwise Accuracy: {correct_predictions}/{len(test_samples)} = {100 * correct_predictions / len(test_samples):.2f}%")
print("=" * 100)

# Retrieval Testing
print("\nRetrieval Task Testing...")
retrieval_samples = random.sample(all_data, min(10, len(all_data)))

for idx, sample in enumerate(retrieval_samples, 1):
  query = sample['query']
  pos_doc = sample['pos']

  # Sample negative candidates
  random_negs = random.sample([s['neg'] for s in all_data if s != sample], min(4, len(all_data)-1))
  candidates = [pos_doc] + random_negs
  random.shuffle(candidates)

  correct_idx = candidates.index(pos_doc)

  query_emb = model.encode(query, convert_to_tensor=False)
  cand_embs = model.encode(candidates, convert_to_tensor=False)

  # Convert to tensors for cosine similarity
  query_tensor = torch.tensor(query_emb)
  cand_tensor = torch.tensor(cand_embs)

  similarities = util.cos_sim(query_tensor, cand_tensor)[0]
  ranked_indices = similarities.argsort(descending=True).numpy()
  correct_rank = np.where(ranked_indices == correct_idx)[0][0] + 1

  print(f"\nRetrieval Test {idx}:")
  print(f"Correct Document Rank: {correct_rank}/{len(candidates)}")
  print(f"Top-1: {'✓' if correct_rank == 1 else '✗'}")

  print("Top-3:")
  for rank, i in enumerate(ranked_indices[:3], 1):
    print(f" {rank}. sim={similarities[i]:.4f} {'← correct' if i == correct_idx else ''}")
  print("-" * 90)

print("\nTesting completed! 🚀")

Starting fine-tuning evaluation...


Sample 1/40
Positive: 0.8836, Negative: 0.0052
Result: ✓ Correct
------------------------------------------------------------------------------------------

Sample 2/40
Positive: 0.7760, Negative: -0.0524
Result: ✓ Correct
------------------------------------------------------------------------------------------

Sample 3/40
Positive: 0.8126, Negative: 0.2996
Result: ✓ Correct
------------------------------------------------------------------------------------------

Sample 4/40
Positive: 0.7417, Negative: -0.1425
Result: ✓ Correct
------------------------------------------------------------------------------------------

Sample 5/40
Positive: 0.7156, Negative: 0.3208
Result: ✓ Correct
------------------------------------------------------------------------------------------

Sample 6/40
Positive: 0.4373, Negative: 0.0201
Result: ✓ Correct
------------------------------------------------------------------------------------------

Sample 7/40
Positiv